# Structural Design Patterns

## Quick notes

### What this covers
Patterns for assembling classes and objects into larger structures: Adapter, Bridge, Composite, Decorator (the pattern vs Python's `@decorator`), Facade, Flyweight (with `__slots__` and `weakref`), and Proxy (virtual, protection, caching, and `__getattr__` forwarding), plus the Python-idiomatic alternative for each.

### Key points
- **Adapter**: wraps a class with the *wrong* interface so your code can use it. In Python often just a function, lambda, or small subclass.
- **Bridge**: two things vary independently (shape x color), so keep two hierarchies and let one *hold* the other. N + M classes instead of N x M.
- **Composite**: leaf and group share one interface, so a single file and a whole folder tree are treated the same. Groups forward calls to their children.
- **Decorator pattern**: wrap an object in another with the *same interface* to add behavior at runtime; wrappers stack in any order (Espresso -> Milk -> Sugar).
- **Python `@decorator`**: a language feature that wraps functions/classes at definition time. Related idea, not the same thing. Use `functools.wraps`.
- **Facade**: one simple entry point over a messy subsystem; often just a module or function in Python. The subsystem stays usable directly.
- **Flyweight**: share the *intrinsic* (unchanging) state across many objects via a factory cache; pass *extrinsic* (per-use) state in. `__slots__` shrinks each object; a `WeakValueDictionary` cache cleans itself.
- **Proxy**: same interface, but it *controls access*: **virtual** (lazy load), **protection** (permission check), **caching** (remember results).
- **`__getattr__` proxies** forward unknown attribute lookups to a target, so you don't hand-write every method.
- **Python alternatives**: duck typing (Adapter), passing a callable (Bridge), recursion over nested data (Composite), `@decorator`/`contextlib` (Decorator), a module of functions (Facade), `lru_cache`/interning/`__slots__` (Flyweight), `cached_property`/`__getattr__` (Proxy).

### Gotchas (the part worth revising twice)
- All the wrapper patterns look alike in code; **intent** differs. Adapter changes the interface, Decorator adds behavior, Proxy controls access, Facade simplifies many objects.
- Without `functools.wraps`, a decorated function loses its `__name__` and docstring.
- Composite: operations like `add()` make no sense on a leaf — decide where they live.
- `__slots__` blocks new attributes (typos raise `AttributeError`) and removes `__weakref__` unless you list it in the slots, which matters for weak caches.
- `__getattr__` only runs when normal lookup **fails**, and dunder methods (`len(proxy)`, `str(proxy)`) are looked up on the *type*, so they are **not** forwarded — define them explicitly.
- A plain dict flyweight cache keeps objects alive forever; use `weakref.WeakValueDictionary` if they should expire.
- A caching proxy serves stale data if the real answer changes; only cache pure results.
- Don't stack adapters on adapters or build deep decorator towers — debugging which layer did what gets painful. A facade that does everything becomes a god class.

### When it matters
Use them when integrating code you can't change (Adapter, Facade), when features combine in many ways (Bridge, Decorator), when data is a tree (Composite), when millions of similar objects strain memory (Flyweight), or when access to an object needs gating, delaying or caching (Proxy). In Python, check first whether a function, a module, or a stdlib helper already does the job.

### One-line recap example
```python
class LazyImage:
    def __init__(self, path): self.path, self._real = path, None
    def show(self):
        self._real = self._real or f"loaded {self.path}"   # virtual proxy: load on first use
        return self._real
```

In [1]:
import functools
import sys
import weakref
from abc import ABC, abstractmethod

## 01. Structural Design Patterns - Introduction

- Structural patterns are about HOW classes and objects are put together
  into bigger structures, while keeping the pieces flexible.
- Covered here: Adapter, Bridge, Composite, Decorator (pattern),
  Facade, Flyweight, Proxy.
- Each section says what problem it solves, when NOT to use it, and a demo.
- Python note: duck typing, first-class functions and __getattr__ make
  several of these much lighter than in Java/C++.

In [2]:
print("Structural patterns: how to assemble objects")
print("Most of them are 'wrap something' in a smart way")

Structural patterns: how to assemble objects
Most of them are 'wrap something' in a smart way


## 02. Adapter - What and Why

- Problem: you have a class with the WRONG interface (third-party or old
  code you cannot edit) but your code expects a different one.
- Fix: a small wrapper that translates calls from one to the other.
- When NOT to use: if you own the class, just change it. Do not stack
  adapters on adapters; that is a sign the design needs a rethink.

In [3]:
class OldPrinter:                       # cannot edit this
    def print_text_upper(self, s):
        return s.upper()

class Printer(ABC):                     # the interface our code expects
    @abstractmethod
    def show(self, s): ...

class PrinterAdapter(Printer):
    def __init__(self, old):
        self._old = old
    def show(self, s):
        return self._old.print_text_upper(s)

def client(printer: Printer):
    return printer.show("hello")

print("adapter:", client(PrinterAdapter(OldPrinter())))

adapter: HELLO


## 03. Adapter - Python Idioms

- Because of duck typing, an adapter is often just a function or a
  lambda; no interface class needed.
- Another trick: a thin subclass that adds the missing method name.
- Stdlib example: functools.partial and operator.attrgetter adapt calls.

In [4]:
old = OldPrinter()
show = lambda s: old.print_text_upper(s)         # function adapter

class AliasedPrinter(OldPrinter):                # subclass adapter
    show = OldPrinter.print_text_upper

print("function adapter:", show("abc"), "| subclass adapter:", AliasedPrinter().show("xyz"))

function adapter: ABC | subclass adapter: XYZ


## 04. Bridge - What and Why

- Problem: two things vary independently (shape x color, remote x device).
  With inheritance you get a class explosion: RedCircle, BlueCircle,
  RedSquare, BlueSquare ... (N x M classes).
- Fix: keep the two hierarchies apart, and let one HOLD the other
  (composition). Now it is N + M classes.
- When NOT to use: only one dimension varies. A plain subclass is fine.

In [5]:
class Color(ABC):
    @abstractmethod
    def fill(self): ...

class Red(Color):
    def fill(self): return "red"

class Blue(Color):
    def fill(self): return "blue"

class ShapeBase:
    def __init__(self, color: Color):       # the bridge
        self.color = color

class Circle(ShapeBase):
    def draw(self): return f"{self.color.fill()} circle"

class Square(ShapeBase):
    def draw(self): return f"{self.color.fill()} square"

for shape in (Circle(Red()), Circle(Blue()), Square(Red())):
    print("bridge:", shape.draw())

bridge: red circle
bridge: blue circle
bridge: red square


## 05. Bridge - Remote and Device Example

- Same idea with "abstraction" (what the user does) and "implementation"
  (the device doing the work). New remotes and new devices combine freely.

In [6]:
class Device(ABC):
    @abstractmethod
    def power(self, on): ...

class Tv(Device):
    def power(self, on): return "TV " + ("on" if on else "off")

class Radio(Device):
    def power(self, on): return "Radio " + ("on" if on else "off")

class Remote:
    def __init__(self, device: Device):
        self.device = device
    def toggle(self):
        self.on = not getattr(self, "on", False)
        return self.device.power(self.on)

class SleepRemote(Remote):             # extends the remote side only
    def sleep(self):
        return self.device.power(False)

print(Remote(Tv()).toggle(), "|", Remote(Radio()).toggle(), "|", SleepRemote(Tv()).sleep())

TV on | Radio on | TV off


## 06. Composite - What and Why

- Problem: you have a tree (folders and files, menus and items) and want
  to treat one item and a group of items the SAME way.
- Fix: leaf and group share one interface; a group holds children and
  forwards the call to each of them.
- When NOT to use: flat data. Also be careful with leaf-only operations
  like add(); they make no sense on a leaf.

In [7]:
class Node(ABC):
    @abstractmethod
    def size(self): ...

class File(Node):
    def __init__(self, name, kb):
        self.name, self.kb = name, kb
    def size(self):
        return self.kb

class Folder(Node):
    def __init__(self, name):
        self.name = name
        self.children = []
    def add(self, node):
        self.children.append(node)
        return self
    def size(self):
        return sum(child.size() for child in self.children)

root = Folder("root").add(File("a.txt", 10)).add(
    Folder("docs").add(File("b.pdf", 200)).add(File("c.pdf", 300)))
print("composite total size:", root.size(), "KB")

composite total size: 510 KB


## 07. Composite - Printing the Tree and Iterating

- A recursive method walks any depth. A generator keeps it lazy.

In [8]:
def walk(node, depth=0):
    yield "  " * depth + node.name
    for child in getattr(node, "children", []):
        yield from walk(child, depth + 1)

print("\n".join(walk(root)))

root
  a.txt
  docs
    b.pdf
    c.pdf


## 08. Decorator Pattern - What and Why

- Problem: add extra behavior to an object at RUNTIME, without changing
  its class and without a subclass for every combination.
- Fix: wrap the object in another object with the SAME interface. Wrappers
  can be stacked in any order.
- When NOT to use: if the extras are fixed, just write one class. Deep
  stacks are hard to debug (which layer did what?).

In [9]:
class Coffee(ABC):
    @abstractmethod
    def cost(self): ...
    @abstractmethod
    def label(self): ...

class Espresso(Coffee):
    def cost(self): return 3
    def label(self): return "espresso"

class AddOn(Coffee):                      # base wrapper, same interface
    def __init__(self, inner: Coffee):
        self.inner = inner

class Milk(AddOn):
    def cost(self): return self.inner.cost() + 1
    def label(self): return self.inner.label() + " + milk"

class Sugar(AddOn):
    def cost(self): return self.inner.cost() + 0.5
    def label(self): return self.inner.label() + " + sugar"

order = Sugar(Milk(Espresso()))
print("decorator pattern:", order.label(), "=", order.cost())

decorator pattern: espresso + milk + sugar = 4.5


## 09. Decorator Pattern vs Python @decorator

- The GoF Decorator pattern wraps OBJECTS to add behavior at runtime.
- Python's @decorator syntax wraps FUNCTIONS or CLASSES at definition time.
- They share the "wrap to add behavior" idea, but they are not the same
  thing. @decorator is a language feature; the pattern is a design idea.
- Always use functools.wraps so the wrapped function keeps its name/docs.

In [10]:
def shout(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper() + "!"
    return wrapper

@shout
def greet(name):
    """Say hi."""
    return f"hi {name}"

print("@decorator:", greet("sam"), "| name kept:", greet.__name__, "| doc:", greet.__doc__)

@decorator: HI SAM! | name kept: greet | doc: Say hi.


## 10. Facade

- Problem: a subsystem has many classes and a fussy call order. Callers
  only need a few common tasks.
- Fix: one simple class with a few easy methods that call the messy parts.
- It does NOT hide the subsystem; advanced users can still use it directly.
- When NOT to use: if the subsystem is already simple. A facade that
  grows into a "god class" doing everything is a smell.

In [11]:
class Amplifier:
    def on(self): return "amp on"
class Projector:
    def on(self): return "projector on"
class Lights:
    def dim(self): return "lights dim"

class HomeTheater:
    def __init__(self):
        self.amp, self.proj, self.lights = Amplifier(), Projector(), Lights()
    def watch_movie(self):
        return [self.lights.dim(), self.proj.on(), self.amp.on()]

print("facade:", HomeTheater().watch_movie())

facade: ['lights dim', 'projector on', 'amp on']


## 11. Facade - Python Idioms

- In Python a facade is often just a module whose functions hide a few
  internal modules, or one friendly function.
- Real examples: requests.get() hides sockets/TLS/HTTP; shutil.copytree()
  hides directory walking; json.dumps() hides the encoder setup.

In [12]:
def quick_report(data):
    """Facade function: three internal steps, one call."""
    cleaned = [x for x in data if x is not None]
    total = sum(cleaned)
    return f"n={len(cleaned)} total={total}"

print("function facade:", quick_report([1, None, 2, 3]))

function facade: n=3 total=6


## 12. Flyweight - What and Why

- Problem: you need MANY objects that mostly hold the same data (the
  letters in a text editor, trees in a game), and memory runs out.
- Fix: split state. Intrinsic (shared, never changes) lives in a shared
  flyweight. Extrinsic (differs per use, like position) is passed in.
- A factory returns the same flyweight for the same key.
- When NOT to use: few objects, or little shared data. Complexity for
  nothing.

In [13]:
class TreeType:                          # intrinsic: shared
    def __init__(self, name, color):
        self.name, self.color = name, color
    def draw(self, x, y):                # extrinsic: passed in
        return f"{self.color} {self.name} at ({x},{y})"

class TreeFactory:
    _types = {}
    @classmethod
    def get(cls, name, color):
        key = (name, color)
        if key not in cls._types:
            cls._types[key] = TreeType(name, color)
        return cls._types[key]

forest = [(TreeFactory.get("oak", "green"), x, 0) for x in range(1000)]
forest.append((TreeFactory.get("pine", "dark green"), 5, 5))
print("trees:", len(forest), "| shared types:", len(TreeFactory._types))
print(forest[3][0].draw(forest[3][1], forest[3][2]))

trees: 1001 | shared types: 2
green oak at (3,0)


## 13. Flyweight - __slots__

- Normal objects keep attributes in a per-object __dict__ (extra memory).
- __slots__ replaces that dict with fixed slots: smaller objects, and
  typos like p.zz = 1 raise AttributeError.
- Trade-off: no new attributes, no default __weakref__ unless you list it.
- Sizes below are compared with sys.getsizeof; we print only the booleans
  so output stays the same on every machine.

In [14]:
class PlainPoint:
    def __init__(self, x, y):
        self.x, self.y = x, y

class SlotPoint:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x, self.y = x, y

pp, sp = PlainPoint(1, 2), SlotPoint(1, 2)
plain_total = sys.getsizeof(pp) + sys.getsizeof(pp.__dict__)
print("slots object smaller:", sys.getsizeof(sp) < plain_total)
print("plain has __dict__:", hasattr(pp, "__dict__"), "| slots has __dict__:", hasattr(sp, "__dict__"))
try:
    sp.z = 3
except AttributeError:
    print("slots blocks new attributes")

slots object smaller: True
plain has __dict__: True | slots has __dict__: False
slots blocks new attributes


## 14. Flyweight - weakref Cache

- A plain dict cache keeps flyweights alive forever.
- WeakValueDictionary drops an entry once nobody else uses the object, so
  the cache cleans itself. (Needs a class that supports weak references;
  normal classes do, __slots__ classes need "__weakref__" in slots.)

In [15]:
class Glyph:
    __slots__ = ("char", "__weakref__")
    _cache = weakref.WeakValueDictionary()

    def __init__(self, char):
        self.char = char

    @classmethod
    def get(cls, char):
        g = cls._cache.get(char)
        if g is None:
            g = cls(char)
            cls._cache[char] = g
        return g

g1, g2 = Glyph.get("a"), Glyph.get("a")
print("same flyweight:", g1 is g2, "| cached:", len(Glyph._cache))
del g1, g2
print("cached after release:", len(Glyph._cache))

same flyweight: True | cached: 1
cached after release: 0


## 15. Proxy - What and Why

- Problem: you want to control ACCESS to an object: delay creating it,
  check permissions, cache results, log calls.
- Fix: a stand-in with the SAME interface that decides when and how to
  forward to the real object.
- Difference from Decorator: a decorator ADDS behavior; a proxy CONTROLS
  access. The code looks alike; the intent differs.
- When NOT to use: no real access concern. Then it is just a wrapper.

In [16]:
class Image(ABC):
    @abstractmethod
    def show(self): ...

class RealImage(Image):
    def __init__(self, path):
        print("  loading", path, "(slow)")
        self.path = path
    def show(self):
        return f"showing {self.path}"

print("proxy ideas: virtual, protection, caching - next three sections")

proxy ideas: virtual, protection, caching - next three sections


## 16. Virtual Proxy (lazy loading)

- Delay creating the expensive object until the first real use.

In [17]:
class LazyImage(Image):
    def __init__(self, path):
        self.path, self._real = path, None
    def show(self):
        if self._real is None:
            self._real = RealImage(self.path)
        return self._real.show()

img = LazyImage("cat.png")
print("created proxy, nothing loaded yet")
print(img.show())
print(img.show())                       # second call: no reload

created proxy, nothing loaded yet
  loading cat.png (slow)
showing cat.png
showing cat.png


## 17. Protection Proxy

- Check permission before letting the call through.

In [18]:
class Vault:
    def open(self):
        return "vault opened"

class VaultProxy:
    def __init__(self, vault, user_role):
        self._vault, self._role = vault, user_role
    def open(self):
        if self._role != "admin":
            raise PermissionError(f"role '{self._role}' cannot open vault")
        return self._vault.open()

print(VaultProxy(Vault(), "admin").open())
try:
    VaultProxy(Vault(), "guest").open()
except PermissionError as e:
    print("denied:", e)

vault opened
denied: role 'guest' cannot open vault


## 18. Caching Proxy

- Remember results of slow calls, return the saved one for the same input.
- Python shortcut: functools.lru_cache does exactly this for functions.
- Gotcha: only cache pure results. If the answer changes over time, the
  cache serves stale data.

In [19]:
class SlowService:
    def __init__(self):
        self.calls = 0
    def fetch(self, key):
        self.calls += 1
        return key * 2

class CachingProxy:
    def __init__(self, service):
        self._service, self._cache = service, {}
    def fetch(self, key):
        if key not in self._cache:
            self._cache[key] = self._service.fetch(key)
        return self._cache[key]

svc = SlowService()
proxy = CachingProxy(svc)
print(proxy.fetch(5), proxy.fetch(5), proxy.fetch(6), "| real calls:", svc.calls)

10 10 12 | real calls: 2


## 19. Proxy with __getattr__ (dynamic forwarding)

- Writing every method by hand is tedious. __getattr__ runs only when a
  normal lookup FAILS, so it can forward unknown names to the real object.
- Great for logging proxies that must mimic any class.
- Gotchas: special methods like __len__ or __str__ are looked up on the
  TYPE, so __getattr__ does not catch them. Also set self._target via
  object.__setattr__ style care if you also override __setattr__.

In [20]:
class LoggingProxy:
    def __init__(self, target):
        self._target = target
        self.log = []

    def __getattr__(self, name):
        attr = getattr(self._target, name)       # AttributeError if missing
        if callable(attr):
            def wrapper(*args, **kwargs):
                self.log.append(f"call {name}{args}")
                return attr(*args, **kwargs)
            return wrapper
        return attr

lp = LoggingProxy([3, 1, 2])
lp.append(0)
lp.sort()
print("proxied list:", lp._target, "| log:", lp.log)
try:
    len(lp)
except TypeError:
    print("len() not forwarded: dunder methods skip __getattr__")

proxied list: [0, 1, 2, 3] | log: ['call append(0,)', 'call sort()']
len() not forwarded: dunder methods skip __getattr__


## 20. Decorator vs Proxy vs Adapter vs Facade

- All four "wrap" something. The difference is INTENT:
  Adapter : change the interface to one you need.
  Decorator: keep the interface, add behavior.
  Proxy   : keep the interface, control access.
  Facade  : new, simpler interface over MANY objects.
- Composite and Bridge are about shape (tree / two hierarchies), and
  Flyweight is about memory sharing.

In [21]:
print("Adapter: new interface | Decorator: add | Proxy: control | Facade: simplify")

Adapter: new interface | Decorator: add | Proxy: control | Facade: simplify


## 21. Python-Idiomatic Alternatives

- Adapter    -> a function, lambda, or small subclass (duck typing).
- Bridge     -> pass a strategy/callable in as an argument.
- Composite  -> nested lists/dicts, or one recursive function.
- Decorator  -> @decorator on functions, contextlib, functools.wraps.
- Facade     -> a module with a few friendly functions.
- Flyweight  -> interning, __slots__, functools.lru_cache on a factory.
- Proxy      -> __getattr__, properties, functools.cached_property.
- Rule: reach for the pattern when it removes real pain, not by habit.

In [22]:
class Report:
    def __init__(self):
        self.builds = 0

    @functools.cached_property          # lazy + cached, a mini virtual proxy
    def data(self):
        self.builds += 1
        return [1, 2, 3]

rep = Report()
rep.data
rep.data
print("cached_property built once:", rep.builds)

@functools.lru_cache(maxsize=None)      # a flyweight/caching factory in one line
def make_key(n):
    return ("key", n)

print("lru_cache flyweight:", make_key(1) is make_key(1))

cached_property built once: 1
lru_cache flyweight: True
